In [ ]:
# @title Start FreeFakeStudio
# @markdown ---
# @markdown ### Persistent Colab settings
WORKSPACE_DIR = "/content/drive/MyDrive/FreeFakeStudio"  # @param {type:"string"}
# @markdown Drive folder for app source, ComfyUI, models, cache, and results.
UPDATE_APP = False  # @param {type:"boolean"}
# @markdown Optional: fast-forward update the Drive app copy. No hard reset.
REPAIR_INSTALL = False  # @param {type:"boolean"}
# @markdown Optional: re-check/re-download missing or suspicious files.
# @markdown ---
# @markdown ### Keys file
UPLOAD_KEYS_TXT = False  # @param {type:"boolean"}
# @markdown Upload a `FreeFakeStudio.keys.txt` file now. It will be saved privately to `WORKSPACE_DIR/config/freefakestudio_keys.txt` and reused later.
KEYS_TXT_PATH = ""  # @param {type:"string"}
# @markdown Optional path to an existing key file. Blank uses the saved private keys file in Drive.
# @markdown ---
# @markdown ### Public route
PUBLIC_ROUTE = "Colab proxy"  # @param ["Colab proxy", "ngrok", "Auto"]
# @markdown Colab proxy is the most stable for Gradio queue streaming. Use ngrok only when you need an external public link.
# @markdown ---
# @markdown ### Direct token overrides
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}
# @markdown Optional override. Blank uses the keys txt, saved private settings, or Colab Secrets where supported.
# @markdown ---
# @markdown ### FLUX.2 Klein text encoder
FLUX_ENCODER = "Custom"  # @param ["Official", "Custom"]
# @markdown Official is the memory-tested FP4 encoder. Custom is used only after a compatible file URL is supplied below.
FLUX_CUSTOM_ENCODER_URL = "https://huggingface.co/ponpoke/flux2-klein-4b-uncensored-text-encoder/resolve/main/flux2-klein-4b-uncensored-q4_k_m.gguf"  # @param {type:"string"}
# @markdown Hugging Face file URL ending in .gguf or .safetensors. Recommended: Q4, 2.0-2.7 GiB. Hard limit: 3.0 GiB.
HUGGINGFACE_TOKEN = ""  # @param {type:"string"}
# @markdown Optional token for a gated repository whose terms you have accepted. It is not written to diagnostics.
# @markdown ---
# @markdown ### Avatar Studio APIs
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown Used by Auto Gallery prompt planning, reference validation, and prompt repair. Blank falls back to Colab Secrets or saved private settings.
TAVILY_API_KEY = ""  # @param {type:"string"}
# @markdown Used by Auto Gallery reference search. Blank falls back to Colab Secrets or saved private settings.
GEMINI_MODEL = ""  # @param {type:"string"}
# @markdown Optional. Leave blank to auto-pick an available Gemini Flash model for your key.
# @markdown ---
# @markdown ### Avatar Gallery search controls
AVATAR_REFERENCE_DOMAINS = "instagram.com"  # @param {type:"string"}
# @markdown Comma-separated domains for Tavily. Use `instagram.com` for the fashion-reference workflow, or blank for open web.
AVATAR_REFERENCE_TIME_RANGE = "month"  # @param ["", "day", "week", "month", "year"]
# @markdown Optional Tavily recency filter.
AVATAR_SEARCH_ROUNDS = 3  # @param {type:"integer"}
# @markdown More rounds improve the chance of reaching the requested gallery count but use more API calls. Range enforced: 1-5.
AVATAR_GALLERY_RETRIES = 2  # @param {type:"integer"}
# @markdown Prompt-repair retries per gallery image after SmolVLM rejects it. Range enforced: 0-3.
AVATAR_MAX_CANDIDATE_DOWNLOADS = 60  # @param {type:"integer"}
# @markdown Max downloaded candidate references per search round. Range enforced: 10-120.
# @markdown ---
# @markdown ### Avatar image-to-text controls
AVATAR_VISION_MAX_EDGE = 768  # @param [768, 1024, 1536]
# @markdown SmolVLM inference resize. 768 is safest with FLUX on a free T4; 1024/1536 are slower and heavier.
AVATAR_VISION_MAX_TOKENS = 900  # @param {type:"integer"}
# @markdown Max SmolVLM answer length for face/body specs and validation.
PROJECT_REPO_URL = "https://github.com/itskrishnamalhotra-stack/FreeFakeStudio.git"  # @param {type:"string"}
# @markdown Use your modified fork/repo here. The original upstream should not overwrite these changes.
# @markdown ---

import os
import re
import subprocess
from pathlib import Path

from google.colab import drive
from IPython.display import HTML, display

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

workspace = Path(WORKSPACE_DIR).expanduser().resolve()
workspace.mkdir(parents=True, exist_ok=True)
app_dir = workspace / "app"
private_config_dir = workspace / "config"
private_config_dir.mkdir(parents=True, exist_ok=True)
saved_keys_path = private_config_dir / "freefakestudio_keys.txt"

display(HTML(f"""
<div style='font-family:Inter,system-ui,sans-serif;max-width:680px;padding:14px 16px;border:1px solid #d0d7de;border-radius:10px'>
  <b>FreeFakeStudio</b><br>
  Google Drive mounted. Workspace: <code>{workspace}</code>
</div>
"""))

def run(args, check=True):
    result = subprocess.run(args, text=True, capture_output=True)
    if check and result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout)[-1500:])
    return result

def parse_keys_txt(text):
    values = {}
    for raw_line in text.splitlines():
        line = raw_line.strip().lstrip("\ufeff")
        if not line or line.startswith(("#", "//", ";")):
            continue
        if line.lower().startswith("export "):
            line = line[7:].strip()
        if "=" in line:
            key, value = line.split("=", 1)
        elif ":" in line:
            key, value = line.split(":", 1)
        else:
            continue
        key = re.sub(r"[^A-Za-z0-9_]", "", key.strip()).upper()
        value = value.strip().strip('"').strip("'")
        if key and value:
            values[key] = value
    aliases = {
        "FFS_NGROK_AUTHTOKEN": "NGROK_AUTH_TOKEN",
        "NGROK_AUTHTOKEN": "NGROK_AUTH_TOKEN",
        "FFS_PUBLIC_ROUTE": "PUBLIC_ROUTE",
        "HF_TOKEN": "HUGGINGFACE_TOKEN",
        "HUGGING_FACE_TOKEN": "HUGGINGFACE_TOKEN",
        "FFS_FLUX_ENCODER_MODE": "FLUX_ENCODER",
        "FFS_FLUX_CUSTOM_ENCODER_URL": "FLUX_CUSTOM_ENCODER_URL",
        "FFS_GEMINI_MODEL": "GEMINI_MODEL",
        "FFS_AVATAR_REFERENCE_DOMAINS": "AVATAR_REFERENCE_DOMAINS",
        "FFS_AVATAR_REFERENCE_TIME_RANGE": "AVATAR_REFERENCE_TIME_RANGE",
        "FFS_AVATAR_SEARCH_ROUNDS": "AVATAR_SEARCH_ROUNDS",
        "FFS_AVATAR_GALLERY_RETRIES": "AVATAR_GALLERY_RETRIES",
        "FFS_AVATAR_MAX_CANDIDATE_DOWNLOADS": "AVATAR_MAX_CANDIDATE_DOWNLOADS",
        "FFS_AVATAR_VISION_MAX_EDGE": "AVATAR_VISION_MAX_EDGE",
        "FFS_AVATAR_VISION_MAX_TOKENS": "AVATAR_VISION_MAX_TOKENS",
    }
    for old_key, new_key in aliases.items():
        if old_key in values and new_key not in values:
            values[new_key] = values[old_key]
    return values

def load_keys_txt():
    if UPLOAD_KEYS_TXT:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("UPLOAD_KEYS_TXT is enabled, but no keys txt file was uploaded.")
        name, data = next(iter(uploaded.items()))
        saved_keys_path.write_bytes(data)
        try:
            saved_keys_path.chmod(0o600)
        except OSError:
            pass
        print(f"Saved uploaded keys file to {saved_keys_path} ({name}).")
    candidate = Path(KEYS_TXT_PATH).expanduser() if KEYS_TXT_PATH.strip() else saved_keys_path
    if candidate.exists():
        return parse_keys_txt(candidate.read_text(encoding="utf-8"))
    return {}

key_file_values = load_keys_txt()

def key_value(name, form_value="", default=None):
    manual = str(form_value or "").strip()
    if manual:
        return manual
    return str(key_file_values.get(name, default or "") or "").strip()

def key_setting(name, form_value, default=""):
    manual = str(form_value or "").strip()
    default_text = str(default or "").strip()
    if name in key_file_values and (not manual or manual == default_text):
        return str(key_file_values[name]).strip()
    return manual or default_text

def key_int(name, form_value, default):
    value = key_setting(name, form_value, default)
    try:
        return int(value)
    except (TypeError, ValueError):
        return int(default)

if not (app_dir / "launch.py").exists():
    if app_dir.exists() and any(app_dir.iterdir()):
        raise RuntimeError(
            f"{app_dir} exists but launch.py is missing. Move/rename it or set a new WORKSPACE_DIR."
        )
    run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(app_dir)])
elif UPDATE_APP:
    dirty = run(["git", "-C", str(app_dir), "status", "--porcelain"], check=False).stdout.strip()
    if dirty:
        stamp = __import__("datetime").datetime.now().strftime("%Y%m%d_%H%M%S")
        stash_msg = f"FreeFakeStudio Colab auto-stash before update {stamp}"
        run(["git", "-C", str(app_dir), "stash", "push", "-u", "-m", stash_msg])
        print(f"Drive app copy had local changes; saved them in git stash: {stash_msg}")
    run(["git", "-C", str(app_dir), "pull", "--ff-only"])

if not (app_dir / "launch.py").exists():
    raise RuntimeError("FreeFakeStudio app copy is incomplete: launch.py is missing.")

os.environ["FFS_WORKSPACE"] = str(workspace)
os.environ["FFS_UPDATE"] = "1" if UPDATE_APP else ""
os.environ["FFS_REPAIR"] = "1" if REPAIR_INSTALL else ""
os.environ["FFS_PUBLIC_ROUTE"] = key_setting("PUBLIC_ROUTE", PUBLIC_ROUTE, "Colab proxy")
os.environ["FFS_NGROK_AUTHTOKEN"] = key_value("NGROK_AUTH_TOKEN", NGROK_AUTH_TOKEN)
os.environ["FFS_FLUX_ENCODER_MODE"] = key_setting("FLUX_ENCODER", FLUX_ENCODER, "Official").lower()
os.environ["FFS_FLUX_CUSTOM_ENCODER_URL"] = key_setting("FLUX_CUSTOM_ENCODER_URL", FLUX_CUSTOM_ENCODER_URL, "https://huggingface.co/ponpoke/flux2-klein-4b-uncensored-text-encoder/resolve/main/flux2-klein-4b-uncensored-q4_k_m.gguf")
hf_token = key_value("HUGGINGFACE_TOKEN", HUGGINGFACE_TOKEN)
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
gemini_key = key_value("GEMINI_API_KEY", GEMINI_API_KEY)
if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key
tavily_key = key_value("TAVILY_API_KEY", TAVILY_API_KEY)
if tavily_key:
    os.environ["TAVILY_API_KEY"] = tavily_key
gemini_model = key_value("GEMINI_MODEL", GEMINI_MODEL)
if gemini_model:
    os.environ["FFS_GEMINI_MODEL"] = gemini_model
os.environ["FFS_AVATAR_REFERENCE_DOMAINS"] = key_setting("AVATAR_REFERENCE_DOMAINS", AVATAR_REFERENCE_DOMAINS, "instagram.com")
os.environ["FFS_AVATAR_REFERENCE_TIME_RANGE"] = key_setting("AVATAR_REFERENCE_TIME_RANGE", AVATAR_REFERENCE_TIME_RANGE, "month")
os.environ["FFS_AVATAR_SEARCH_ROUNDS"] = str(key_int("AVATAR_SEARCH_ROUNDS", AVATAR_SEARCH_ROUNDS, 3))
os.environ["FFS_AVATAR_GALLERY_RETRIES"] = str(key_int("AVATAR_GALLERY_RETRIES", AVATAR_GALLERY_RETRIES, 2))
os.environ["FFS_AVATAR_MAX_CANDIDATE_DOWNLOADS"] = str(key_int("AVATAR_MAX_CANDIDATE_DOWNLOADS", AVATAR_MAX_CANDIDATE_DOWNLOADS, 60))
os.environ["FFS_AVATAR_VISION_MAX_EDGE"] = str(key_int("AVATAR_VISION_MAX_EDGE", AVATAR_VISION_MAX_EDGE, 768))
os.environ["FFS_AVATAR_VISION_MAX_TOKENS"] = str(key_int("AVATAR_VISION_MAX_TOKENS", AVATAR_VISION_MAX_TOKENS, 900))

exec(compile((app_dir / "launch.py").read_text(), str(app_dir / "launch.py"), "exec"))
